# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Unit of analysis for this baseline:** one row = one page for the month (client_hash_id +
content_hash_id), aggregated from March 2026's daily rows -- not a single page-day. A single
day's impressions/clicks are too noisy to act on; the monthly total is what a reviewer would
actually look at.

**The rule, in plain words:** A page is worth reviewing if (1) it has enough impressions this
month to trust its CTR at all, and (2) its CTR falls meaningfully below what pages at its own
position tier typically get. Rank by how big that gap is, weighted by how much traffic is
riding on it.

**Reason code (one, since this is one rule):** `ctr_below_position_expectation`

**Action label:** `review_title_meta`

Two signals this rule leans on, checked below before any of this gets coded:
1. **CTR vs. position** -- flag-linked: this is the same signal behind FlyRank's CTR-fix logic.
2. **Volume (impressions)** -- flag-linked: this is the same signal behind FlyRank's
   quick-win flag -- low-volume pages produce noisy CTR that shouldn't be trusted.

In [1]:
!pip install -q duckdb

import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

con = duckdb.connect()
HF_TOKEN = userdata.get("HF_TOKEN")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

month_03 = con.sql(f"""
    SELECT *
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
    WHERE gsc_data_available IS TRUE
""").df()

print("month_03 (availability-filtered) shape:", month_03.shape)

# Aggregate daily rows up to one row per page for the month -- this is the real unit
# a reviewer acts on, not a single noisy day.
page_month = month_03.groupby(["client_hash_id", "content_hash_id"], as_index=False).agg(
    total_impressions=("gsc_impressions", "sum"),
    total_clicks=("gsc_clicks", "sum"),
    # weighted average position, weighted by daily impressions
    position_weighted_sum=("gsc_avg_position", lambda s: np.nan),  # placeholder, fixed below
    ga4_engaged_sessions=("ga4_engaged_sessions", "sum"),
    sessions_organic=("sessions_organic", "sum"),
    days_with_data=("report_date", "count"),
)

# Proper impression-weighted average position (can't do a weighted mean inside .agg cleanly)
weighted_pos = month_03.groupby(["client_hash_id", "content_hash_id"]).apply(
    lambda g: np.average(g["gsc_avg_position"], weights=g["gsc_impressions"].clip(lower=1))
).reset_index(name="avg_position")

page_month = page_month.drop(columns=["position_weighted_sum"]).merge(
    weighted_pos, on=["client_hash_id", "content_hash_id"]
)

page_month["ctr_pct"] = np.where(
    page_month["total_impressions"] > 0,
    page_month["total_clicks"] / page_month["total_impressions"] * 100,
    np.nan
)

print("page_month shape (one row per page, March):", page_month.shape)
page_month.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

month_03 (availability-filtered) shape: (3611061, 31)


/tmp/ipykernel_3480/914008105.py:35: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weighted_pos = month_03.groupby(["client_hash_id", "content_hash_id"]).apply(


page_month shape (one row per page, March): (176738, 9)


,client_hash_id,content_hash_id,total_impressions,total_clicks,ga4_engaged_sessions,sessions_organic,days_with_data,avg_position,ctr_pct
0,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,0,0,1,9.000000,0.00000
1,client_0797ff3a1fc9a6a5,content_04c67f3541177192,331,2,0,0,31,14.377644,0.60423
2,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,33,0,0,0,6,9.363636,0.00000
3,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,145,0,0,0,30,8.124138,0.00000
4,client_0797ff3a1fc9a6a5,content_1207efddce873942,461,0,0,0,31,14.488069,0.00000


### Signal check 1 of 2: CTR vs. position (flag-linked -- CTR-fix logic)

*One bucket table, n printed, one-word verdict.*

In [7]:
page_month["position_tier"] = pd.cut(
    page_month["avg_position"], bins=[0, 3, 10, 20, 1000],
    labels=["1-3", "4-10", "11-20", "21+"]
)

signal_1 = page_month.groupby("position_tier", observed=True).agg(
    n=("ctr_pct", "count"),
    median_ctr=("ctr_pct", "median"),
).reset_index()
print(signal_1)

# VERDICT (fill in after running, based on the real table above):
# CONFIRMED if median CTR clearly decays as position tier gets worse;
# MIXED if the direction is right but not monotonic; OPPOSITE / FALSE otherwise.
verdict_signal_1 = "TODO: CONFIRMED / OPPOSITE / MIXED / FALSE -- fill in from the table above"
print("\nVerdict:", verdict_signal_1)

  position_tier      n  median_ctr
0           1-3  17426         0.0
1          4-10  83288         0.0
2         11-20  29922         0.0
3           21+  44668         0.0

Verdict: TODO: CONFIRMED / OPPOSITE / MIXED / FALSE -- fill in from the table above


### Signal check 2 of 2: volume (flag-linked -- quick-win)

*One bucket table, n printed, one-word verdict.*

Low-volume pages should show noisier (more zero-click) CTR than high-volume pages -- if not,
gating the rule on a minimum-impressions threshold isn't actually buying anything.

In [8]:
volume_bins = [0, 100, 500, 2000, 10000, np.inf]
volume_labels = ["<100", "100-500", "500-2000", "2000-10000", "10000+"]
page_month["volume_bucket"] = pd.cut(page_month["total_impressions"], bins=volume_bins, labels=volume_labels)

signal_2 = page_month.groupby("volume_bucket", observed=True).agg(
    n=("ctr_pct", "count"),
    zero_ctr_share=("ctr_pct", lambda s: (s == 0).mean()),
).reset_index()
print(signal_2)

# VERDICT (fill in after running, based on the real table above):
# CONFIRMED if zero_ctr_share clearly drops as volume increases.
verdict_signal_2 = "TODO: CONFIRMED / OPPOSITE / MIXED / FALSE -- fill in from the table above"
print("\nVerdict:", verdict_signal_2)

  volume_bucket      n  zero_ctr_share
0          <100  75506        0.930986
1       100-500  39356        0.681802
2      500-2000  32012        0.292921
3    2000-10000  23987        0.055864
4        10000+   5877        0.009529

Verdict: TODO: CONFIRMED / OPPOSITE / MIXED / FALSE -- fill in from the table above


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Transparent score, no fitted weights: volume gate (0/1) x impressions x the CTR gap
(clipped at zero so underperforming pages score above overperforming ones). Everything
in this formula is knowable from the same March data checked above -- no future window,
no label-derived input.

In [12]:
MIN_IMPRESSIONS = 500  # from Signal 2's bucket table -- below this, CTR is too noisy to trust

visible = (page_month["total_impressions"] >= MIN_IMPRESSIONS).astype(int)

# IMPORTANT: the position-tier baseline must be computed ONLY from visible (volume-gated)
# pages. Signal 1 showed median CTR is 0.0 in every tier when computed on the full,
# unfiltered set -- that's zero-inflation from low-volume noise, not a real baseline.
# Using an unfiltered median here would make ctr_gap negative-or-zero for every single row,
# silently zeroing out the whole score.
visible_rows = page_month[visible == 1]
tier_baseline = visible_rows.groupby("position_tier", observed=True)["ctr_pct"].median()
print("Position-tier baseline, computed on visible pages only:")
print(tier_baseline)

expected_ctr_by_tier = page_month["position_tier"].astype(str).map(tier_baseline.to_dict()).astype(float)
page_month["ctr_gap"] = expected_ctr_by_tier - page_month["ctr_pct"]  # positive = underperforming

positive_gap = page_month["ctr_gap"].clip(lower=0)

page_month["score"] = visible * page_month["total_impressions"] * positive_gap
page_month["reason_code"] = "ctr_below_position_expectation"
page_month["action"] = "review_title_meta"

ranked = page_month.sort_values("score", ascending=False).reset_index(drop=True)

import os
os.makedirs("work/outputs", exist_ok=True)

output_cols = [
    "client_hash_id", "content_hash_id", "avg_position", "position_tier",
    "total_impressions", "total_clicks", "ctr_pct", "ctr_gap",
    "score", "reason_code", "action"
]
ranked[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print("\nRows written:", len(ranked))
print("Rows with score > 0 (visible + underperforming):", (ranked["score"] > 0).sum())
ranked[output_cols].head(10)

Position-tier baseline, computed on visible pages only:
position_tier
1-3      0.235942
4-10     0.214592
11-20    0.166667
21+      0.075268
Name: ctr_pct, dtype: float64

Rows written: 176738
Rows with score > 0 (visible + underperforming): 30958


,client_hash_id,content_hash_id,avg_position,position_tier,total_impressions,total_clicks,ctr_pct,ctr_gap,score,reason_code,action
0,client_23a62021009f63c4,content_44f34c0a90047651,0.665877,1-3,212404,24,0.011299,0.224643,47714.990054,ctr_below_position_expectation,review_title_meta
1,client_73cda7b4e4f265ea,content_8e1334d6356668e3,2.693038,1-3,134984,1,0.000741,0.235201,31748.372994,ctr_below_position_expectation,review_title_meta
2,client_73cda7b4e4f265ea,content_fec55986a1868d62,0.308426,1-3,124075,1,0.000806,0.235136,29174.483489,ctr_below_position_expectation,review_title_meta
3,client_62f4a7e64f5e0096,content_34a70fea29d15f24,3.166132,4-10,143019,43,0.030066,0.184526,26390.772532,ctr_below_position_expectation,review_title_meta
4,client_62f4a7e64f5e0096,content_f6116743b00afc2d,9.735658,4-10,107584,15,0.013943,0.200650,21586.695279,ctr_below_position_expectation,review_title_meta
5,client_62f4a7e64f5e0096,content_7c6373141eae744a,5.948459,4-10,132593,83,0.062598,0.151995,20153.433476,ctr_below_position_expectation,review_title_meta
6,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,0.116003,1-3,83834,1,0.001193,0.234749,19679.948006,ctr_below_position_expectation,review_title_meta
7,client_e547b89c05043229,content_8d7d99f109e19aa2,2.468557,1-3,203497,289,0.142017,0.093925,19113.456107,ctr_below_position_expectation,review_title_meta
8,client_9958f0a7ae1df715,content_cd3d932d4e1c8db0,7.831807,4-10,89332,4,0.004478,0.210115,18769.957082,ctr_below_position_expectation,review_title_meta
9,client_a80fca3f171ed1de,content_046fc480045b88f5,7.208276,4-10,83788,6,0.007161,0.207431,17380.257511,ctr_below_position_expectation,review_title_meta


## 3. Top-10 review

*For each of your top ten, one line each — the action, why it's there, and what would make it wrong.*

For each row: read `total_impressions`, `avg_position`, and `ctr_gap` before writing the line --
a generic template isn't a review. Replace the placeholder text below with what the actual
numbers say once this cell has real output.

In [13]:
top10 = ranked[output_cols].head(10).copy()
print(top10.to_string(index=False))

# For each of the 10 rows above, write one line:
# "<content_hash_id> -- review_title_meta, because <impressions> impressions at position
#  <avg_position> only converts at <ctr_pct>% vs. its tier's expected rate (gap: <ctr_gap>).
#  Would be wrong if: <e.g. position is unstable across the month / this is a seasonal page /
#  the low CTR is a snippet issue an editor already fixed after March>."
#
# TODO: fill in 10 real lines here, one per row in top10, using the actual printed numbers.

         client_hash_id          content_hash_id  avg_position position_tier  total_impressions  total_clicks  ctr_pct  ctr_gap        score                    reason_code            action
client_23a62021009f63c4 content_44f34c0a90047651      0.665877           1-3             212404            24 0.011299 0.224643 47714.990054 ctr_below_position_expectation review_title_meta
client_73cda7b4e4f265ea content_8e1334d6356668e3      2.693038           1-3             134984             1 0.000741 0.235201 31748.372994 ctr_below_position_expectation review_title_meta
client_73cda7b4e4f265ea content_fec55986a1868d62      0.308426           1-3             124075             1 0.000806 0.235136 29174.483489 ctr_below_position_expectation review_title_meta
client_62f4a7e64f5e0096 content_34a70fea29d15f24      3.166132          4-10             143019            43 0.030066 0.184526 26390.772532 ctr_below_position_expectation review_title_meta
client_62f4a7e64f5e0096 content_f6116743b00afc2d  

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Leakage check (structural, can state now):**
- No FlyRank product flag (`health_score`, `priority_score`, etc.) was used anywhere in the
  score -- not in this dataset by design.
- No future window: every column feeding the score comes from `month=2026-03` only; nothing
  from April onward or from the June `_sample`/test table touched this rule.
- No label-derived input: `gsc_clicks` only enters through `ctr_pct`, which the score compares
  against a same-month position-tier baseline -- it isn't reused as an independent "feature"
  the way Section 3's trap in w03 warned against.

**Weak picks (data-dependent -- fill in after Section 3's real top-10):** look for rows where
a high score comes from a position that's unstable across the month (check daily variance,
not just the monthly average) or from a page that only had traffic for a few days out of the
month -- both would make a "confirmed" gap look bigger than it really is.

In [14]:
# Quick structural leakage check: assert none of the excluded/leak-prone columns
# ended up feeding the score.
leak_prone = ["health_score", "priority_score", "gsc_sum_position"]
present_leak_cols = [c for c in leak_prone if c in ranked.columns]
print("Leak-prone columns present in ranked output (should be empty):", present_leak_cols)

# Check for position instability within the month for the top 10 -- a high day-to-day
# swing in gsc_avg_position means the monthly average is less trustworthy.
top10_ids = ranked.head(10)[["client_hash_id", "content_hash_id"]]
daily_for_top10 = month_03.merge(top10_ids, on=["client_hash_id", "content_hash_id"])
position_stability = daily_for_top10.groupby(
    ["client_hash_id", "content_hash_id"]
)["gsc_avg_position"].agg(["mean", "std", "count"]).reset_index()
print("\nPosition stability across March for the top 10 (high std = less trustworthy pick):")
print(position_stability)

Leak-prone columns present in ranked output (should be empty): []

Position stability across March for the top 10 (high std = less trustworthy pick):
            client_hash_id           content_hash_id       mean        std  \
0  client_23a62021009f63c4  content_44f34c0a90047651   7.346909   4.565167   
1  client_62f4a7e64f5e0096  content_34a70fea29d15f24   3.219473   0.830142   
2  client_62f4a7e64f5e0096  content_7c6373141eae744a   5.789019   0.987275   
3  client_62f4a7e64f5e0096  content_f6116743b00afc2d   9.536301   0.800204   
4  client_73cda7b4e4f265ea  content_8e1334d6356668e3   4.545582   2.576839   
5  client_73cda7b4e4f265ea  content_9c057b66c30a3abb  11.195379  13.556438   
6  client_73cda7b4e4f265ea  content_fec55986a1868d62   9.385150   9.811790   
7  client_9958f0a7ae1df715  content_cd3d932d4e1c8db0   7.786219   0.613971   
8  client_a80fca3f171ed1de  content_046fc480045b88f5   7.289152   0.895849   
9  client_e547b89c05043229  content_8d7d99f109e19aa2   2.563756   0.20

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.